# Licensing recommendation analysis

This notebook records the analysis plan for Kwame Boateng’s fictional $14M back-catalog licensing decision. The plan will turn the profile and exploratory analysis into a recommendation Adjoa can rerun each quarter. See the [lab brief](../data/lab_brief.md) and [decision record](../docs/DECISIONS.md).

## Decision questions

1. Does the evidence favor franchise bundles or standalone films, and is that conclusion stable across the required time windows?
2. Which eligible franchises and standalone titles merit consideration in each window, and what supports their inclusion?
3. Do directors or leading-cast members show repeatable associations with stronger outcomes after relevant context is considered?
4. What relative-value case can be made for the strongest franchise, and which KPI most strongly supports it?

## Expected deliverables

- A franchise-versus-standalone comparison reported separately for all years and 2000 onward.
- Side-by-side shortlists targeting three franchises and five standalones per window, with reasons, evidence limits, and rank changes.
- A cautious director and leading-cast signal analysis with role definitions, title counts, and uncertainty.
- A one-paragraph deal note for the strongest franchise and a one-page client memo covering the recommendation, shortlists, both windows, and limitations.

## Interpretation guardrails

Historical box office measures theatrical performance. Popularity changes over time and overlaps with vote count. Genre, talent, and collection patterns are observational. Keep plausible extremes in the primary analysis, and highlight any change in the recommendation between windows.

## What has been done so far

### Data-quality profile — `02_profile_movie_data.ipynb`

- The normalized tables and relationship grains were profiled for the current 112-movie cache.
- No exact raw-object duplicates, exact duplicate movie rows, or title/release-year semantic duplicate candidates were found. Tested range and date checks passed, while the profile also records missing reference labels, constant fields, and narrow language coverage.
- A copy-only dtype experiment preserved the normalized movie frame. Its memory result applies only to this small snapshot; runtime was not benchmarked.

### Exploratory analysis — `03_eda_movie_data.ipynb`

- Numeric distributions, IQR-flagged observations, and plausible extreme values were investigated. Popularity remains snapshot-sensitive.
- Associations were observed between budget and revenue, popularity and vote count, and vote count and rating. These are not causal findings; popularity and vote count overlap.
- Genre, collection, director, and top-billed-cast relationships were explored. Common genre indicators did not improve held-out R-squared over the budget/year/collection controls in this sample. Collection and talent residuals remain descriptive and sample-sensitive.

### Candidate coverage so far

The 127 input rows reconcile to 112 distinct cached titles, 6 repeated rows, 5 TMDB 404s, 3 invalid IDs, and 1 blank. Of the 112 cached titles, 71 were released in 2000 or later. The full pool has 16 titles without a collection ID, and the newer window has 8, before scoring eligibility. See the [findings log](../docs/findings.md).

**Current status**

The profile and EDA are complete for this snapshot. The two-window comparison, score calculation, shortlists, and memo remain to be produced.

## What is next

### 1. Reconcile inputs and apply the two release windows

- Reconcile every row of the current `id_list.csv` to a cache or fetch outcome. Intersect cached titles with the valid unique IDs in the current list so a previous quarter's cache entries cannot enter the candidate pool. Keep one row per `movie_id`. Report source losses, titles outside the window, and KPI eligibility exclusions separately.
- Use dated, released titles in **A — all years** and only dated, released titles with `release_date >= 2000-01-01` in **B — 2000 onward**. Report missing dates and release statuses. Every summary and score in B uses only its in-window titles. Both windows use the same scoring constants, eligibility gates, and tie-break rules. The windows overlap, so agreement measures sensitivity to the cutoff.
- Report counts by year or decade, title type, and collection, then compare outcome coverage and missingness. Use release year or era for context in nominal financial comparisons. The cutoff addresses Yaw's concern and leaves the historical dollar values nominal.

### 2. Use a fixed audience-first scoring rule

Use vote-adjusted audience rating as the primary shortlist KPI. It measures how TMDB voters rated a title, with fewer-vote ratings pulled toward a common reference. Vote count measures the breadth of that evidence and also serves as the reach-first sensitivity. Retain popularity as context. An audience-first shortlist could favor well-liked films with limited reach, which the reach-first comparison should surface.

- For **title eligibility**, require a finite `vote_average` in [0, 10] and an integer `vote_count` of at least 1,000. Report missing and invalid vote metrics, and titles below the threshold, as separate reasons. Apply financial validity rules to the corresponding KPIs.
- Calculate the **title score** as `(vote_count * vote_average + 1000 * 7.0) / (vote_count + 1000)`. Use 7.0 as a conservative point on the rating scale for this pool, whose exploratory mean is about 7.59. Make observed votes and prior votes count equally at the 1,000-vote entry threshold. These fixed lab assumptions apply across both windows and reruns. Use the full-precision score to rank and round for display.
- Order **standalones** by adjusted rating descending, vote count descending, then numeric `movie_id` ascending. A missing collection ID defines a standalone candidate in the available metadata.
- For **franchise eligibility and ranking**, group by `collection_id` within the window. Require at least two observed dated, released installments and score every such installment to qualify. A known in-window installment with unusable rating evidence makes the proxy bundle ineligible. Rank by median adjusted rating descending, lowest installment adjusted rating descending, median vote count descending, then numeric `collection_id` ascending. Each observed installment gets equal weight. List the observed titles and flag unknown coverage beyond the candidate list. Use these observed titles as the hypothetical lab package.
- For the **shortlists**, take up to three eligible collections and five eligible standalones in order, and explain any shortfall. Calculate positive-budget, positive-gross financial proxies where defined: reported gross minus budget, and `(revenue - budget) / budget`. Log zeros separately from missing values. Show these as context for historical performance, with the valid-film count. TMDB provides no platform outcomes or acquisition costs from which to calculate licensing return.

### 3. Compare bundles and standalones at both levels

- Compare like decision units first. Each eligible proxy bundle contributes its median adjusted rating once, and each eligible standalone contributes its adjusted rating once. Report the gap in group medians in rating points, the spread, and the number of eligible units and films. Also show a per-film comparison of all score-eligible collection-linked titles and standalones, so the effect of bundle eligibility and large collections is visible.
- If each side has at least five eligible units, sort decision units by ID within each side. Use NumPy’s default random generator with seed 42 and draw 2,000 bootstrap resamples within each side. Take the 2.5th and 97.5th percentiles of the bundle-minus-standalone median gaps for a descriptive 95% interval. The primary comparison favors bundles if the interval lies entirely above +0.25 rating points, or standalones if it lies entirely below −0.25. Otherwise report insufficient evidence to favor either. The 0.25-point margin and minimum group size are lab judgments. The interval reflects variation within this candidate list.
- Retain plausible extreme values in the primary analysis. Show bundle size, lowest installment score, valid-financial-film counts, and observed gross totals alongside per-film medians. Report the impact of removing the highest-scoring installment from each bundle (lowest numeric movie ID in a score tie) in a sensitivity run, including any bundle that drops below two films.

### 4. Review genre and talent signals

- Count titles per genre and report genre overlaps. Use the EDA cutoffs in both windows. Require at least 10 titles for descriptive comparisons and 15 for genre indicators in an adjusted model. Examine relationships with audience response as well as exploratory log gross. Use budget, release year, genre, and collection status as context where the sample supports it.
- Identify directors by crew job `Director` and use cast order 0–2 as the leading-cast proxy. Deduplicate on `(movie_id, person_id, role)`, group people and collections by ID, and attach names for display. Report distinct titles, distinct collections, and distinct standalone titles. Count each collection as one group and each standalone as its own group. Require at least three distinct titles across at least two groups for a cross-title signal. Flag one-franchise concentrations even when the title threshold is met.
- Compare adjusted outcomes by person and check how the direction changes when the person's largest group by title count is omitted. Break size ties by group type (collection before standalone) and then numeric group ID. If fewer than two titles remain, or the direction flips or disappears, label the signal fragile and show why. Report changes in eligibility and signal between windows. Use release-time and collection-aware holdouts if making any predictive claim. With sparse data, keep the result descriptive and avoid causal language.

### 5. Stress-test and make the rankings reproducible

- Rerun both windows with a reach-first order. Lead with vote count for titles and median vote count for bundles, then use the remaining primary rules in order. Also rerun the primary ranking with vote thresholds of 500 and 5,000 and, separately, with a 6.5 rating anchor. Report who enters, leaves, or moves and whether the leading franchise changes. Recompute the group recommendation for each rating-anchor, vote-threshold, and installment-omission sensitivity. Use nominal per-film gross as a separate comparison with its known era and budget limitations.
- Produce one full candidate table per unit and window, including exclusions and reasons. Show raw and adjusted rating, votes, popularity, financial proxies, year and genre, observed package membership, rank, evidence for inclusion, and the main limitation. Compare shortlist membership and ranks by ID side by side. Label the recommendation and material signals **stable**, **changed**, or **insufficient evidence**.
- Implement the rule once and call it for both windows. Use vectorized checks and aggregations. Record the input and cache fingerprints, code revision, cache timestamp range, parameters, counts at each gate, and elapsed fetch and analysis times separately. Persist terminal fetch failures by ID and reason so Adjoa can distinguish a failed request from a missing TMDB title. Reuse the cached snapshot for an exact rerun. A fresh TMDB fetch can change the evidence. Check ranking ties, missing values, source-list changes, and top-to-bottom execution, then measure a cached run at approximately 500 titles.

### 6. Write the deal note and client memo

- Name the leading eligible lab bundle and its strongest supporting number with the window, denominator, and main limitation. Compare it with the next-best bundle and the strongest standalone alternative. Explain which bundle Kwame should negotiate for first and highlight any change in the leading franchise between windows. Use historical gross and production budget only to describe past theatrical performance. The lab supplies no licensing quotes, so Kwame will need prices to tell what fits within his $14M budget.
- Write one page in business language with the recommendation, both shortlists and their reasons, the two-window comparison, the deal note, and the cost of the remaining unknowns. Point Adjoa to the run record as the proposed improvement beyond the brief. It lets her reproduce a result and explain changes at the next review.